In [24]:
import torch

def find_tensor_sub_seq(batch_ids, sub_seq_ids):    
    first_token = sub_seq_ids[0]
    batch_size, seq_len = batch_ids.shape
    sub_seq_len = len(sub_seq_ids)
    positions = torch.full((batch_size,), seq_len, dtype=torch.int, device=batch_ids.device)
    for b in range(batch_size):
        seq = batch_ids[b]
        candidate_positions = (seq == first_token).nonzero(as_tuple=True)[0]
        for pos in candidate_positions:
            if pos + sub_seq_len <= seq_len:
                window = seq[pos:pos+sub_seq_len]
                if torch.all(window == sub_seq_ids):
                    positions[b] = pos
                    break
    return positions




input_ids = torch.Tensor([[1, 2, 3, 4, 4, 4, 6, 9], [5, 6, 2, 3, 4, 4, 6, 9]])
sor_token_ids = torch.Tensor([6, 9])
eor_token_ids = torch.Tensor([2, 3])

sor_positions = find_tensor_sub_seq(input_ids, sor_token_ids)
eor_positions = find_tensor_sub_seq(input_ids, eor_token_ids)
prompt_complete_flags = (sor_positions < eor_positions).unsqueeze(1).repeat(1, input_ids.shape[1])
end_prompt_mask = (torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0) 
                    <= (eor_positions + len(eor_token_ids) - 1).unsqueeze(1))
start_prompt_mask = (torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0) 
                   >= (sor_positions).unsqueeze(1))
prompt_mask = torch.where(prompt_complete_flags, start_prompt_mask & end_prompt_mask, start_prompt_mask | end_prompt_mask)

print(prompt_mask)

# print(eor_prompt_mask)
# print(sor_prompt_mask)
# print(sor_prompt_mask & eor_prompt_mask)

tensor([1, 2], dtype=torch.int32)
tensor([6, 6], dtype=torch.int32)
tensor([[False, False, False, False, False, False, False, False],
        [False, False, False, False, False, False, False, False]])
tensor([[ True,  True,  True, False, False, False,  True,  True],
        [ True,  True,  True,  True, False, False,  True,  True]])
